## RQ2: Inefficiencies in Off-Peak Service

Introduction

Research Question:
Are there inefficiencies (low ridership relative to fixed schedule) in B-Line’s current service, especially during off-peak hours?

This research question aims to uncover scheduling inefficiencies in B-Line’s fixed-route service by examining ridership relative to the frequency of scheduled trips. In particular, we will focus on identifying periods (such as late nights or midday lulls) where buses run frequently despite low demand. This can guide adjustments to improve operational efficiency.

### Variables

**Dependent Variable (Outcome):**

boardings – number of passengers boarding per observation (from RidershipData.csv)

**Independent Variables (Predictors):**

hour – time of day (in RidershipData.csv)

date – calendar date (in RidershipData.csv)

scheduled_trips – number of trips scheduled per time block (from ScheduleData.csv or GTFS feed)

route_id – route identifier (common key for merging across data sources)

These variables come primarily from RidershipData.csv and ScheduleData.csv, which together allow comparison between actual ridership and planned service levels.

## Statistical Plan

**Preprocessing:**

Convert timestamp to datetime

Create time buckets (e.g., hourly)

Aggregate boardings by hour and route

Join with schedule data to get scheduled_trips

Calculate metrics like load factor = boardings / scheduled_trips

Methods, These methods will help identify time periods and routes where service is underutilized, supporting data-driven service adjustments and cost efficiency:
- **Exploratory Data Analysis (EDA)**: Identify low-ridership periods with high schedule frequency

- **Clustering**: Group similar hours/routes based on load factor using K-Means or DBSCAN

- **Threshold Flagging**: Identify off-peak hours with boardings below a defined threshold despite high frequency



In [1]:
import pandas as pd
import numpy as np


In [2]:
validation = pd.read_csv("C:/Users/zelaskar/Box/2024 Zakir Elaskar/data/Combined datasets/validation_combined_union.csv")



C:\Users\zelaskar\AppData\Local\Temp\ipykernel_5676\1892028941.py:1: DtypeWarning: Columns (10,17,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  validation = pd.read_csv("C:/Users/zelaskar/Box/2024 Zakir Elaskar/data/Combined datasets/validation_combined_union.csv")


In [3]:
# Ensure datetime
validation["service_datetime"] = pd.to_datetime(
    validation["timestamp"],
    format="mixed",
    errors="coerce"
)


In [4]:
validation["service_datetime"].isna().sum()


np.int64(0)

In [5]:
# Check that trips are identified by trip_id
validation["trip_id"].nunique()


9469

In [6]:
# Inspect a few trip identifiers and related fields
validation[["trip_id", "route_id", "timestamp"]].dropna().head(10)


,trip_id,route_id,timestamp
0,719d625e-da62-4604-9bc8-86e57a0a6ce8,3.0,1/4/2023 9:00
1,1a60c296-b304-4c7d-b0ed-4084143eefe1,3.0,1/17/2023 11:29
2,719d625e-da62-4604-9bc8-86e57a0a6ce8,3.0,1/21/2023 11:11
3,9d46f116-ea4f-4fe9-929e-e58134210070,3.0,1/30/2023 14:05
4,aabc0a92-a0d2-4542-8335-d2f704c846c3,3.0,1/30/2023 14:05
5,229f0343-da33-49e8-9326-b1bc24e50080,3.0,1/31/2023 11:36
6,f3573f30-c4c5-4a63-b1f4-d7001b286c2b,20.0,1/9/2023 16:36
7,6a82b14e-c1ca-4cf8-b41d-c485afb54ae9,20.0,1/9/2023 16:36
8,6a82b14e-c1ca-4cf8-b41d-c485afb54ae9,20.0,1/26/2023 7:15
9,f3573f30-c4c5-4a63-b1f4-d7001b286c2b,20.0,1/31/2023 7:13


In [7]:
# Number of boarding events per trip (rows per trip)
validation.groupby("trip_id").size().describe()


count    9469.000000
mean       13.827014
std        22.920775
min         1.000000
25%         2.000000
50%         6.000000
75%        15.000000
max       398.000000
dtype: float64

## Data Preparation and Time Variable Construction

The validation dataset records one row per boarding event. 

Timestamps were converted to datetime objects to extract temporal information used in the analysis.


In [8]:
validation["service_datetime"] = pd.to_datetime(
    validation["timestamp"],
    format="mixed",
    errors="coerce"
)

validation = validation.dropna(subset=["service_datetime"])

validation["hour"] = validation["service_datetime"].dt.hour
validation["day_of_week"] = validation["service_datetime"].dt.day_name()


## Boardings per Trip

Service utilization is measured at the trip level as boardings per trip, defined as the total number of passenger boarding events observed on a single bus trip. Because the boarding event–level dataset records one row per passenger boarding, boardings per trip are computed by grouping boarding records by trip identifier and counting the number of boarding events associated with each trip.

This trip-level measure represents passenger usage per bus run and serves as the dependent variable in subsequent analyses of service utilization.



In [9]:
# Compute boardings per trip:
# Each row in df = one boarding event
trip_counts = (
    validation.groupby("trip_id")
      .size()
      .reset_index(name="boardings_per_trip")
)

In [10]:
# Attach route and time information to each trip
# (Using the first timestamp per trip as the trip start)
trip_meta = (
    validation.sort_values("service_datetime")
      .groupby("trip_id", as_index=False)
      .agg(
          route_id=("route_id", "first"),
          hour=("hour", "first"),
          day_of_week=("day_of_week", "first"),
          trip_start_time=("service_datetime", "first")
      )
)

In [11]:
# Final trip-level dataset for regression
trip_regression = trip_meta.merge(
    trip_counts, on="trip_id", how="inner"
)

# Inspect result
trip_regression.head()

,trip_id,route_id,hour,day_of_week,trip_start_time,boardings_per_trip
0,0002cb24-fc47-45f7-bae8-ed2acec5e054,5,11,Friday,2024-06-07 11:20:00,4
1,00059f62-1536-4ede-9d82-1ff84189ed9b,7,17,Monday,2025-01-06 17:43:00,3
2,0007e49a-8ba3-4d4c-afaf-0d07d7194796,2,14,Sunday,2024-05-19 14:27:00,37
3,000bca34-308a-4af0-b282-70ec5316e57f,5,13,Saturday,2023-09-02 13:23:00,2
4,00196df3-cf2d-4eed-b79a-a74813f6e6e7,15,14,Saturday,2024-11-30 14:13:00,3


Prior to model estimation, categorical predictors were explicitly encoded and route identifiers were standardized to ensure consistent factor levels. Basic distributional and diagnostic checks were performed to verify the structure of the trip-level dataset before fitting the regression model.


In [12]:
# Ensure correct types
trip_regression["hour"] = trip_regression["hour"].astype("category")
trip_regression["day_of_week"] = trip_regression["day_of_week"].astype("category")
trip_regression["route_id"] = trip_regression["route_id"].astype("category")


In [13]:
# Distribution of utilization
trip_regression["boardings_per_trip"].describe()


count    9469.000000
mean       13.827014
std        22.920775
min         1.000000
25%         2.000000
50%         6.000000
75%        15.000000
max       398.000000
Name: boardings_per_trip, dtype: float64

In [14]:
# Continuous summary
cont_summary = trip_regression['boardings_per_trip'].describe()

# Convert to DataFrame and round
cont_table = cont_summary.to_frame(name='Boardings per Trip')

cont_table = cont_table.round(2)

cont_table

,Boardings per Trip
count,9469.00
mean,13.83
std,22.92
min,1.00
25%,2.00
50%,6.00
75%,15.00
max,398.00


In [15]:
# Check categories
trip_regression["hour"].unique()
trip_regression["day_of_week"].value_counts()


day_of_week
Friday       2402
Tuesday      1823
Monday       1801
Wednesday    1261
Saturday     1119
Thursday      937
Sunday        126
Name: count, dtype: int64

In [16]:
# Make a clean copy
trip_regression = trip_regression.copy()

# Route id: convert to string and remove trailing ".0" if present
trip_regression["route_id"] = (
    trip_regression["route_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

# Hour: ensure it's an integer 0–23
trip_regression["hour"] = trip_regression["hour"].astype(int)

# Day of week: ensure it's a string (or category)
trip_regression["day_of_week"] = trip_regression["day_of_week"].astype(str)


In [17]:
sorted(trip_regression["route_id"].unique())[:30]


['14',
 '15',
 '16',
 '17',
 '2',
 '20',
 '24',
 '25',
 '26',
 '27',
 '3',
 '30',
 '32',
 '4',
 '40',
 '41',
 '5',
 '52',
 '7',
 '8',
 '9',
 '9C',
 'nan']

## Multivariable linear regression
To examine how service utilization varies across the service schedule without imposing predefined peak or off-peak periods, a multivariable regression model is estimated using boardings per trip as the outcome variable. Hour of day and day of week are included as categorical predictors to capture non-linear temporal variation in utilization, and route identifiers are included to control for persistent differences in ridership across routes.


In [18]:
import statsmodels.formula.api as smf

# Make copies
dfm = trip_regression.copy()

# Set reference categories (choose what you like)
dfm["hour"] = pd.Categorical(dfm["hour"], ordered=False)
dfm["day_of_week"] = pd.Categorical(
    dfm["day_of_week"],
    categories=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"],
    ordered=True
)

# Refit with explicit reference categories
rq2_model_ref = smf.ols(
    formula="""
    boardings_per_trip
    ~ C(hour, Treatment(reference=10))
    + C(day_of_week, Treatment(reference='Monday'))
    + C(route_id)
    """,
    data=dfm
).fit(cov_type="HC3")


rq2_model_ref.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     boardings_per_trip   R-squared:                       0.136
Model:                            OLS   Adj. R-squared:                  0.132
Method:                 Least Squares   F-statistic:                     22.07
Date:                Wed, 04 Mar 2026   Prob (F-statistic):          5.36e-162
Time:                        14:53:59   Log-Likelihood:                -42176.
No. Observations:                9415   AIC:                         8.444e+04
Df Residuals:                    9371   BIC:                         8.475e+04
Df Model:                          43                                         
Covariance Type:                  HC3                                         
==============================================================================================================================
                                                                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------------------
Intercept                                                     23.8352      1.753     13.597      0.000      20.400      27.271
C(hour, Treatment(reference=10))[T.5]                         59.8658     21.011      2.849      0.004      18.684     101.047
C(hour, Treatment(reference=10))[T.6]                         12.4064      2.162      5.737      0.000       8.168      16.644
C(hour, Treatment(reference=10))[T.7]                         10.3142      1.492      6.914      0.000       7.390      13.238
C(hour, Treatment(reference=10))[T.8]                          0.7461      1.007      0.741      0.459      -1.227       2.719
C(hour, Treatment(reference=10))[T.9]                         -0.5708      0.888     -0.643      0.520      -2.311       1.170
C(hour, Treatment(reference=10))[T.11]                        -1.1562      0.845     -1.368      0.171      -2.813       0.500
C(hour, Treatment(reference=10))[T.12]                         1.2989      0.976      1.331      0.183      -0.614       3.212
C(hour, Treatment(reference=10))[T.13]                         1.4726      1.019      1.445      0.148      -0.524       3.469
C(hour, Treatment(reference=10))[T.14]                         4.4794      1.241      3.609      0.000       2.047       6.912
C(hour, Treatment(reference=10))[T.15]                         3.1033      1.069      2.902      0.004       1.007       5.199
C(hour, Treatment(reference=10))[T.16]                         2.8482      1.032      2.759      0.006       0.825       4.871
C(hour, Treatment(reference=10))[T.17]                         0.3741      1.024      0.365      0.715      -1.633       2.381
C(hour, Treatment(reference=10))[T.18]                        -4.7097      0.895     -5.263      0.000      -6.464      -2.956
C(hour, Treatment(reference=10))[T.19]                        -5.6846      0.936     -6.075      0.000      -7.518      -3.851
C(hour, Treatment(reference=10))[T.20]                        -7.6123      0.908     -8.386      0.000      -9.391      -5.833
C(hour, Treatment(reference=10))[T.21]                        -8.0214      1.560     -5.143      0.000     -11.078      -4.965
C(day_of_week, Treatment(reference='Monday'))[T.Tuesday]      -3.8933      0.894     -4.355      0.000      -5.646      -2.141
C(day_of_week, Treatment(reference='Monday'))[T.Wednesday]    -6.7043      0.873     -7.679      0.000      -8.415      -4.993
C(day_of_week, Treatment(reference='Monday'))[T.Thursday]     -7.0074      0.914     -7.663      0.000      -8.800      -5.215
C(day_of_week, Treatment(reference='Monday'))[T.Friday]      -12.4482      0.756    -16.459      0.000     -13.931     -10.966
C(day_of_week, Treatment(reference=

In [24]:
summary_table = rq2_model_ref.summary2().tables[1]

# Keep only relevant predictors (exclude route effects if desired)
clean_table = summary_table.copy()

# Optional: drop route fixed effects from main table
clean_table = clean_table[~clean_table.index.str.contains("route_id")]

# Rename columns for publication clarity
clean_table = clean_table.rename(columns={
    "Coef.": "Coefficient",
    "Std.Err.": "Std. Error",
    "P>|z|": "p-value"
})

clean_table.round(3)

,Coefficient,Std. Error,z,p-value,[0.025,0.975]
Intercept,23.835,1.753,13.597,0.000,20.400,27.271
"C(hour, Treatment(reference=10))[T.5]",59.866,21.011,2.849,0.004,18.684,101.047
"C(hour, Treatment(reference=10))[T.6]",12.406,2.162,5.737,0.000,8.168,16.644
"C(hour, Treatment(reference=10))[T.7]",10.314,1.492,6.914,0.000,7.390,13.238
"C(hour, Treatment(reference=10))[T.8]",0.746,1.007,0.741,0.459,-1.227,2.719
"C(hour, Treatment(reference=10))[T.9]",-0.571,0.888,-0.643,0.520,-2.311,1.170
"C(hour, Treatment(reference=10))[T.11]",-1.156,0.845,-1.368,0.171,-2.813,0.500
"C(hour, Treatment(reference=10))[T.12]",1.299,0.976,1.331,0.183,-0.614,3.212
"C(hour, Treatment(reference=10))[T.13]",1.473,1.019,1.445,0.148,-0.524,3.469
"C(hour, Treatment(reference=10))[T.14]",4.479,1.241,3.609,0.000,2.047,6.912


In [22]:
# --- Add 95% Confidence Intervals (from robust HC3 model) ---
ci = rq2_model_ref.conf_int()
ci.columns = ["CI Lower (95%)", "CI Upper (95%)"]

# Merge CI into your existing clean_table
final_table = clean_table.join(ci)

# Round numeric values
final_table = final_table.round(3)

# Keep only publication-relevant columns
final_table = final_table[[
    "Coefficient",
    "Std. Error",
    "CI Lower (95%)",
    "CI Upper (95%)",
    "p-value"
]]

# Export publication-ready regression table
final_table.to_csv("Table2_RQ2_Regression_Results.csv")
final_table.to_excel("Table2_RQ2_Regression_Results.xlsx")

final_table.head()

,Coefficient,Std. Error,CI Lower (95%),CI Upper (95%),p-value
Intercept,23.835,1.753,20.400,27.271,0.000
"C(hour, Treatment(reference=10))[T.5]",59.866,21.011,18.684,101.047,0.004
"C(hour, Treatment(reference=10))[T.6]",12.406,2.162,8.168,16.644,0.000
"C(hour, Treatment(reference=10))[T.7]",10.314,1.492,7.390,13.238,0.000
"C(hour, Treatment(reference=10))[T.8]",0.746,1.007,-1.227,2.719,0.459


In [20]:
# Convert .ipynb to .qmd (no --to flag needed)
#!quarto convert RQ2.ipynba

In [21]:
#!quarto render RQ2.ipynb --to pdf --log-level debug


In [25]:
summary_table = rq2_model_ref.summary2().tables[1]

# Keep all predictors (INCLUDING route effects)
clean_table = summary_table.copy()

# Rename columns for publication clarity
clean_table = clean_table.rename(columns={
    "Coef.": "Coefficient",
    "Std.Err.": "Std. Error",
    "P>|z|": "p-value"
})

clean_table.round(3)

,Coefficient,Std. Error,z,p-value,[0.025,0.975]
Intercept,23.835,1.753,13.597,0.000,20.400,27.271
"C(hour, Treatment(reference=10))[T.5]",59.866,21.011,2.849,0.004,18.684,101.047
"C(hour, Treatment(reference=10))[T.6]",12.406,2.162,5.737,0.000,8.168,16.644
"C(hour, Treatment(reference=10))[T.7]",10.314,1.492,6.914,0.000,7.390,13.238
"C(hour, Treatment(reference=10))[T.8]",0.746,1.007,0.741,0.459,-1.227,2.719
"C(hour, Treatment(reference=10))[T.9]",-0.571,0.888,-0.643,0.520,-2.311,1.170
"C(hour, Treatment(reference=10))[T.11]",-1.156,0.845,-1.368,0.171,-2.813,0.500
"C(hour, Treatment(reference=10))[T.12]",1.299,0.976,1.331,0.183,-0.614,3.212
"C(hour, Treatment(reference=10))[T.13]",1.473,1.019,1.445,0.148,-0.524,3.469
"C(hour, Treatment(reference=10))[T.14]",4.479,1.241,3.609,0.000,2.047,6.912
